
# Laboratorio 4 — Selección y Extracción de Características
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 26 de agosto de 2026 · Sesión 4 de 17
**Bloques:** 6-8 (11:40-13:40) · Lab. Central 114
**Unidad 1:** Fundamentos y Preparación

**Objetivo:** comparar en código las dos estrategias de la teoría — **seleccionar** features
(quedarse con algunas de las columnas originales) vs **extraer** features (construir columnas
nuevas con PCA) — usando siempre el mismo modelo: **KNN**, elegido a propósito por ser el que
más sufre la maldición de la dimensionalidad.

> **De dónde venimos:** en el Lab 3, KNN escalado llegó a 0.9630 en Wine usando las **13**
> features. Y quedó una pregunta abierta en Titanic: `pclass` y las dummies `class_*` decían
> exactamente lo mismo. Hoy la respondemos — y de paso, la pregunta del Lab 1: *"¿qué pasaría
> con 10.000 features?"*

---
## Parte 1 · Selección: quedarse con las mejores columnas


### 1. Setup y split (código dado)

Wine de siempre, mismo split del Lab 3 (`test_size=0.3`, `random_state=42`) para que los
resultados sean comparables con los de la semana pasada.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split

wine = load_wine(as_frame=True)
X, y = wine.data, wine.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("train:", X_train.shape, "| test:", X_test.shape)


### 2. Rankear las features con `SelectKBest`

`SelectKBest` con `f_classif` aplica la **prueba ANOVA F** de la teoría: para cada feature mide
qué tan distintas son sus medias entre los 3 cultivares. Mucha diferencia → la feature separa
bien las clases → puntaje alto.

**Antes de correr nada, responde:** ¿qué features de Wine esperas que salgan arriba en el
ranking? *(vale intuición de enólogo, o mirar el `describe()` del Lab 1)*

> *(tu respuesta)*

**Tu tarea:**
1. Crea un `SelectKBest(score_func=f_classif, k=5)`.
2. Ajústalo (`fit`) **solo con los datos de train** — misma regla de oro del Lab 3: el ranking
   también se aprende del entrenamiento.
3. Arma una `pd.Series` con `selector.scores_` indexada por `X.columns` y ordénala de mayor a menor.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif


**Responde aquí (doble clic para editar):** ¿cuáles quedaron en el top-5? ¿Coincide con lo que
esperabas? ¿Alguna sorpresa — alguna feature que parecía importante y quedó al fondo?

> *(tu respuesta)*


### 3. Comparar KNN: 13 features vs las top-5

Ahora la pregunta que importa: **¿cuánto se pierde** al tirar 8 de las 13 columnas?

Ojo con el orden de las operaciones: **primero se selecciona, después se escala**. Y el
escalamiento sigue respetando la regla de oro (`fit` solo en train).

**Tu tarea:**
1. **Con todas las features:** escala (`StandardScaler`, fit en train / transform en test),
   entrena un KNN y guarda su score en `score_full`.
2. **Con las top-5:** obtén `X_train_sel` / `X_test_sel` con `selector.transform(...)`, escálalas
   con un `StandardScaler` **nuevo**, entrena otro KNN y guarda su score en `score_sel`.
3. Imprime los dos.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

# 1. todas las features (13)


# 2. solo las top-5


### 4. Discusión

1. ¿El score **bajó, subió, o se mantuvo igual** al usar solo 5 de las 13 features?
2. Si no bajó: ¿por qué sacar columnas no perjudicó — o incluso ayudó? Conecta con la **maldición
   de la dimensionalidad** y con el espacio de hipótesis más chico de la teoría de esta mañana.
3. ¿Habría dado lo mismo escalar **antes** de seleccionar? *(pista: el F-test compara medias entre
   clases — ¿lo cambia una transformación que le aplica lo mismo a todas las filas?)*

> *(tus respuestas)*

---
## ☕ Recreo

---
## Parte 2 · Extracción: construir columnas nuevas con PCA


### 5. PCA sobre Wine, dos componentes

En vez de descartar columnas, PCA **combina las 13 en 2 nuevas** — las direcciones de máxima
varianza de la teoría. Aquí lo usamos sobre **todo Wine** con un único propósito: **visualizar**.
*(Para modelar viene en la sección 8, ahí sí con la regla de oro.)*

**Antes de ver el gráfico, responde:** con solo 2 de las 13 dimensiones originales, ¿esperas que
los 3 cultivares se distingan en el scatter?

> *(tu respuesta)*

**Tu tarea:**
1. Escala **todo** `X` con un `StandardScaler` nuevo (PCA sin escalar es una trampa: la componente
   1 sería casi pura `proline`, solo porque su varianza es enorme).
2. Ajusta un `PCA(n_components=2)` y transforma.
3. Grafica el scatter coloreando por `y`, con ejes rotulados.
4. Imprime `pca.explained_variance_ratio_`.

In [ ]:
from sklearn.decomposition import PCA


**Responde aquí:** ¿cuánta varianza captura cada componente, y cuánta las dos juntas? Es decir,
¿qué fracción de la información original sobrevive en el gráfico que acabas de ver? ¿Te sorprende,
dado lo bien que se separan las clases?

> *(tu respuesta)*


### 6. La maldición de la dimensionalidad, en vivo: Digits

Wine tiene 13 features. **Digits** tiene **64**: cada imagen es un dígito manuscrito de 8×8
píxeles, y cada píxel es una feature. Es el dataset más grande que hemos tocado — y la
oportunidad de ver comprimido lo que en la teoría era un dibujo en la pizarra.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_d, y_d = digits.data, digits.target
print(X_d.shape)   # (1797, 64) — 64 features, una por pixel de la imagen 8x8

# así se ve uno de los dígitos "desarmado" en 64 números:
plt.imshow(X_d[0].reshape(8, 8), cmap="gray")
plt.title(f"un {y_d[0]}, sus 64 features")
plt.show()


**Tu tarea (a):** aplica `PCA(n_components=2)` a `X_d` y grafica el scatter coloreando por `y_d`
(usa `cmap="tab10"`, son 10 clases; agrega `plt.colorbar()` para saber qué color es qué dígito).


**Tu tarea (b):** ajusta un `PCA()` **sin limitar** componentes y grafica la curva de
**varianza explicada acumulada** (`.explained_variance_ratio_.cumsum()`).
*Tip: `plt.axhline(0.9, ls="--")` deja marcada la línea del 90%.*


**Responde aquí:**
1. Con 64 features comprimidas a solo 2, ¿los dígitos igual se agrupan de forma reconocible?
   ¿Cuáles se confunden entre sí, y tiene sentido que esos justamente se confundan?
2. Lee la curva: ¿cuántas componentes hacen falta para llegar al **90%** de la varianza?
   ¿Y al 95%? Compáralo con 64.
3. Ahora vuelve a la pregunta del Lab 1: *"¿y si fueran 10.000 features?"* — con lo que acabas
   de ver, ¿qué le harías a ese dataset antes de entrenar nada?

> *(tus respuestas)*

### 6b. Reconstrucción: ¿cuánto se recupera con pocas componentes? *(si sobra tiempo)*

PCA no solo comprime — con `pca.inverse_transform()` también reconstruye una aproximación del
dato original a partir de las componentes retenidas. Cada componente que se descarta es
información que no vuelve: reconstruir con pocas componentes muestra literalmente cuánto se
pierde.

**Tu tarea — las bases:**
1. Ajusta un `PCA(n_components=5)` sobre **todo** `X_d`.
2. Cada fila de `pca.components_` es una "imagen base" de 64 números — grafica las 5 primeras
   como imágenes de 8×8 (`plt.subplots(1, 5, figsize=(10, 2))`), con `cmap="RdBu"` (un
   componente tiene valores positivos y negativos, no solo grises: rojo y azul son los dos
   signos).

**Tu tarea — la reconstrucción:**
1. Elige un dígito de `X_d` (por ejemplo `X_d[0]`).
2. Para `n_components` en `[1, 2, 5, 10, 20, 64]`: ajusta un `PCA(n_components=n)` sobre
   **todo** `X_d`, transforma el dígito y reconstrúyelo con `.inverse_transform(...)`.
3. Grafica las 6 reconstrucciones en fila (`plt.subplots(1, 6, figsize=(12, 2))`), cada una con
   `imshow(..., cmap="gray")` y el número de componentes como título.

**Responde aquí:** las 5 imágenes base — ¿se parecen a dígitos, o son manchas abstractas? Cada
reconstrucción es una **combinación** (suma pesada) de estas bases: ¿tiene sentido entonces que
con solo 1-2 no alcance para reconocer un dígito?

¿A partir de cuántas componentes la reconstrucción ya es reconocible a simple vista? ¿Coincide
con el codo que viste en la curva de varianza acumulada de la tarea anterior? Con las 64
componentes (todas), ¿la reconstrucción es idéntica al original — por qué?

> *(tu respuesta)*

---
## Parte 3 · Las tres estrategias, cara a cara


### 7. Comparación final: todas vs seleccionadas vs PCA

El mismo KNN, el mismo split, tres versiones de Wine. Esta es la síntesis de toda la clase.

Aquí PCA sí se usa para **modelar**, así que vuelve la regla de oro: `fit_transform` en train,
`transform` (sin `fit`) en test. Se aplica sobre los datos **ya escalados** de la sección 3.

**Tu tarea:** entrena un KNN sobre 2 componentes principales, guarda el score en `score_pca` e
imprime los tres resultados lado a lado.

In [ ]:
# PCA para modelar: fit en train, transform en test (sobre X_train_s / X_test_s de la sección 3)


print(f"Todas las features (13): {score_full:.4f}")
print(f"Seleccionadas (5):       {score_sel:.4f}")
print(f"PCA (2 componentes):     {score_pca:.4f}")


**Responde aquí — la pregunta del día:** con estos tres números en la mano, ¿cuál estrategia
elegirías para Wine? Argumenta considerando el trade-off de la teoría:

- **Selección:** sabes exactamente qué columnas quedaron (`flavanoids`, `proline`, …) y puedes
  explicárselas a un enólogo.
- **Extracción:** más compacta, pero "componente 1" no significa nada que un enólogo entienda.

Y si el PCA de 2 componentes te pareció poco: pruébalo con 3 o 4 en la celda de abajo antes de
opinar.

> *(tu respuesta)*

In [ ]:
# opcional: ¿mejora PCA con más componentes?


### 8. Multicolinealidad: dos columnas que dicen lo mismo *(si sobra tiempo)*

En la teoría vimos que features muy correlacionadas **entre sí** son información duplicada. En el
Lab 3 lo tuvimos delante: `pclass` y `class_*` en Titanic.

**Tu tarea:** calcula la matriz de correlación de Wine (`X.corr()`), y encuentra los pares de
features distintas con mayor correlación absoluta.
*Tip: `X.corr().abs().mask(np.eye(len(X.columns), dtype=bool)).unstack().sort_values(ascending=False)`*

In [ ]:
import numpy as np


**Responde aquí:** ¿qué par salió más correlacionado? ¿Aparecen ambas features en el top-5 que
seleccionaste en la sección 2? Si es así — ¿es un problema que `SelectKBest` haya elegido las dos?
*(pista: el F-test evalúa cada feature **por separado**, nunca mira si dos ya dicen lo mismo)*

> *(tu respuesta)*


### 9. Ejercicio propuesto (no evaluado)

Repite la comparación completa de la sección 7 con `load_breast_cancer` — **30 features** de
partida, más del doble que Wine.

¿La brecha entre las tres estrategias se hace **más grande** o **más chica** al partir con más
features? ¿Por qué crees que pasa eso?

In [ ]:
from sklearn.datasets import load_breast_cancer

# mismo flujo de las secciones 3 y 7, con test_size=0.3 y random_state=42


---
**Moraleja del lab:** reducir features no es solo un truco de eficiencia. Hoy sacarle columnas
al modelo **no lo empeoró** — porque las columnas de más no traían señal, traían ruido. Y siempre
hay un trade-off: compacidad vs poder explicar qué está mirando el modelo.

**Anuncios:**
- Próxima clase (02-sep): **train-test split y validación cruzada** — cierre teórico de la
  Unidad 1. Por fin se explica a fondo ese `train_test_split(..., random_state=42)` que llevamos
  dos labs usando "de prestado" *(¿y si el 42 hubiera sido otro número? ¿cambian todos los scores
  de hoy?)*.
- **Certamen 1: 09-sep**, cubre toda la Unidad 1.
- La **Tarea 1** se enunciará próximamente, condensando las clases 1 a 4.